#### Check if metadata from 2015 is correct to 2012 dataset

In [ ]:
import os

labels_file = "/imagenet/devkit/ILSVRC/devkit/data/ILSVRC2015_clsloc_validation_ground_truth.txt"
val_dir = "/imagenet/val"  # folder with ILSVRC2012_val_*.JPEG

labels = [int(x.strip()) for x in open(labels_file)]
imgs = sorted([f for f in os.listdir(val_dir) if f.endswith(".JPEG")])

print("labels:", len(labels))
print("images:", len(imgs))
print("first img:", imgs[0], "last img:", imgs[-1])
print("label range:", min(labels), max(labels))


labels: 50000
images: 50000
first img: ILSVRC2012_val_00000001.JPEG last img: ILSVRC2012_val_00050000.JPEG
label range: 1 1000


#### Split flat validation dataset into class subfolders

In [ ]:
import os
import shutil

# -----------------------
# PATHS (your setup)
# -----------------------
VAL_IMAGES_DIR = "/imagenet/val"
DEVKIT_DATA_DIR = "/imagenet/devkit/ILSVRC/devkit/data"

LABELS_FILE = os.path.join(DEVKIT_DATA_DIR, "ILSVRC2015_clsloc_validation_ground_truth.txt")
MAP_FILE = os.path.join(DEVKIT_DATA_DIR, "map_clsloc.txt")

OUTPUT_DIR = "/imagenet/val_classed"
MOVE_FILES = True  # set False to copy instead of move

# -----------------------
# Helpers
# -----------------------
def parse_map_clsloc(map_path: str):
    """
    Returns a dict: index (1..1000) -> wnid (e.g., n01440764).

    map_clsloc.txt formats vary slightly across devkits.
    This parser supports the common patterns, e.g.:
      n01440764 1
      1 n01440764
      n01440764 1 tench, Tinca tinca
    """
    idx_to_wnid = {}

    with open(map_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()

            # Try detect which token is the index and which is wnid
            # WNIDs start with 'n' and are like n########
            wnid = None
            idx = None

            for p in parts:
                if p.startswith("n") and len(p) == 9 and p[1:].isdigit():
                    wnid = p
                elif p.isdigit():
                    idx = int(p)

            if wnid is None or idx is None:
                # If your map file is unusual, we fail loudly
                raise ValueError(f"Could not parse line in map_clsloc.txt: {line}")

            # indices should be 1..1000
            idx_to_wnid[idx] = wnid

    if len(idx_to_wnid) != 1000:
        print(f"[WARN] Parsed {len(idx_to_wnid)} classes from map_clsloc.txt (expected 1000).")
    return idx_to_wnid


def main():
    # Load labels (1..1000)
    with open(LABELS_FILE, "r") as f:
        labels = [int(x.strip()) for x in f if x.strip()]

    # Sort images to align with labels
    images = sorted([fn for fn in os.listdir(VAL_IMAGES_DIR) if fn.endswith(".JPEG")])

    if len(images) != len(labels):
        raise RuntimeError(f"Mismatch: {len(images)} images vs {len(labels)} labels. "
                           f"Check VAL_IMAGES_DIR and LABELS_FILE alignment.")

    # Load index -> wnid mapping
    idx_to_wnid = parse_map_clsloc(MAP_FILE)

    # Create output dir + class dirs
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for wnid in set(idx_to_wnid.values()):
        os.makedirs(os.path.join(OUTPUT_DIR, wnid), exist_ok=True)

    moved = 0
    for img_name, label in zip(images, labels):
        wnid = idx_to_wnid[label]  # labels are 1..1000
        src = os.path.join(VAL_IMAGES_DIR, img_name)
        dst = os.path.join(OUTPUT_DIR, wnid, img_name)

        if MOVE_FILES:
            shutil.move(src, dst)
        else:
            shutil.copy2(src, dst)

        moved += 1
        if moved % 5000 == 0:
            print(f"... processed {moved}/{len(images)}")

    print(f"✅ Done. Organized {moved} validation images into:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


... processed 5000/50000
... processed 10000/50000
... processed 15000/50000
... processed 20000/50000
... processed 25000/50000
... processed 30000/50000
... processed 35000/50000
... processed 40000/50000
... processed 45000/50000
... processed 50000/50000
✅ Done. Organized 50000 validation images into:
/data/local/rgaisina/datasets/imagenet/val_classed


#### Create validation manifest using bboxes

In [5]:
import os
import json
import glob
import statistics
import xml.etree.ElementTree as ET

# -----------------------
# PATHS (your setup)
# -----------------------
VAL_CLASSED_DIR = "/data/local/rgaisina/datasets/imagenet/val_classed"
VAL_XML_DIR = "/data/local/rgaisina/datasets/imagenet/bboxes/val"
OUT_META_DIR = "/data/local/rgaisina/datasets/imagenet/meta"

MANIFEST_JSONL = os.path.join(OUT_META_DIR, "val_manifest.jsonl")
BBOX_BY_ID_JSON = os.path.join(OUT_META_DIR, "val_bboxes_by_id.json")
STATS_JSON = os.path.join(OUT_META_DIR, "val_bbox_stats.json")

os.makedirs(OUT_META_DIR, exist_ok=True)


def parse_xml(xml_path: str):
    """
    Returns:
      image_id (str) e.g. ILSVRC2012_val_00000001
      wnid (str) e.g. n01440764
      bboxes_xyxy: List[List[int]] as [xmin, ymin, xmax, ymax] (inclusive-ish; standard ImageNet boxes)
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename_tag = root.find("filename")
    if filename_tag is None or not filename_tag.text:
        raise ValueError(f"Missing <filename> in {xml_path}")
    filename = filename_tag.text.strip()
    image_id = os.path.splitext(filename)[0]  # remove .JPEG

    # WNID is stored in each <object><name> tag; typically same for all objects
    objects = root.findall("object")
    if not objects:
        raise ValueError(f"No <object> entries in {xml_path}")

    wnid = None
    bboxes = []

    for obj in objects:
        name_tag = obj.find("name")
        if name_tag is not None and name_tag.text:
            if wnid is None:
                wnid = name_tag.text.strip()

        bnd = obj.find("bndbox")
        if bnd is None:
            continue

        xmin = int(float(bnd.findtext("xmin", "0")))
        ymin = int(float(bnd.findtext("ymin", "0")))
        xmax = int(float(bnd.findtext("xmax", "0")))
        ymax = int(float(bnd.findtext("ymax", "0")))

        # basic sanity
        if xmax <= xmin or ymax <= ymin:
            continue

        bboxes.append([xmin, ymin, xmax, ymax])

    if wnid is None:
        raise ValueError(f"Could not determine WNID from {xml_path}")

    return image_id, wnid, bboxes


def bbox_area_xyxy(bb):
    xmin, ymin, xmax, ymax = bb
    return max(0, xmax - xmin) * max(0, ymax - ymin)


def find_image_rel_path(wnid: str, image_id: str):
    """
    In val_classed, images are stored as: val_classed/<wnid>/<image_id>.JPEG
    Returns rel_path or None if missing.
    """
    rel_path = os.path.join(wnid, image_id + ".JPEG")
    abs_path = os.path.join(VAL_CLASSED_DIR, rel_path)
    if os.path.exists(abs_path):
        return rel_path
    return None


def main():
    xml_files = sorted(glob.glob(os.path.join(VAL_XML_DIR, "*.xml")))
    if not xml_files:
        raise RuntimeError(f"No XML files found in {VAL_XML_DIR}")

    manifest_lines = []
    bbox_by_id = {}

    n_with_images = 0
    n_missing_images = 0
    n_no_boxes = 0
    bbox_counts = []
    bbox_areas = []

    for xml_path in xml_files:
        image_id, wnid, bboxes = parse_xml(xml_path)

        if len(bboxes) == 0:
            n_no_boxes += 1
            # still record entry with empty boxes (useful for accounting)
            primary = None
        else:
            bbox_counts.append(len(bboxes))
            areas = [bbox_area_xyxy(bb) for bb in bboxes]
            bbox_areas.extend(areas)
            #primary = bboxes[max(range(len(bboxes))),]  # placeholder

            # choose largest-area bbox as primary
            max_i = max(range(len(bboxes)), key=lambda i: bbox_area_xyxy(bboxes[i]))
            primary = bboxes[max_i]

        rel_path = find_image_rel_path(wnid, image_id)
        if rel_path is None:
            n_missing_images += 1
        else:
            n_with_images += 1

        entry = {
            "image_id": image_id,
            "wnid": wnid,
            "rel_path": rel_path,  # may be None if not found
            "bboxes_xyxy": bboxes,
            "primary_bbox_xyxy": primary,
        }

        manifest_lines.append(entry)
        bbox_by_id[image_id] = {
            "wnid": wnid,
            "rel_path": rel_path,
            "bboxes_xyxy": bboxes,
            "primary_bbox_xyxy": primary,
        }

    # Write JSONL manifest
    with open(MANIFEST_JSONL, "w") as f:
        for entry in manifest_lines:
            f.write(json.dumps(entry) + "\n")

    # Write dict JSON
    with open(BBOX_BY_ID_JSON, "w") as f:
        json.dump(bbox_by_id, f, indent=2)

    # Stats
    stats = {
        "num_xml_files": len(xml_files),
        "num_entries": len(manifest_lines),
        "num_with_existing_images_in_val_classed": n_with_images,
        "num_missing_images_in_val_classed": n_missing_images,
        "num_entries_with_zero_valid_boxes": n_no_boxes,
        "bbox_count_per_image": {
            "min": min(bbox_counts) if bbox_counts else None,
            "max": max(bbox_counts) if bbox_counts else None,
            "mean": float(statistics.mean(bbox_counts)) if bbox_counts else None,
            "median": float(statistics.median(bbox_counts)) if bbox_counts else None,
        },
        "bbox_area": {
            "min": min(bbox_areas) if bbox_areas else None,
            "max": max(bbox_areas) if bbox_areas else None,
            "mean": float(statistics.mean(bbox_areas)) if bbox_areas else None,
            "median": float(statistics.median(bbox_areas)) if bbox_areas else None,
        },
    }

    with open(STATS_JSON, "w") as f:
        json.dump(stats, f, indent=2)

    print("✅ Wrote:")
    print("  -", MANIFEST_JSONL)
    print("  -", BBOX_BY_ID_JSON)
    print("  -", STATS_JSON)
    print("Summary:", json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()


✅ Wrote:
  - /data/local/rgaisina/datasets/imagenet/meta/val_manifest.jsonl
  - /data/local/rgaisina/datasets/imagenet/meta/val_bboxes_by_id.json
  - /data/local/rgaisina/datasets/imagenet/meta/val_bbox_stats.json
Summary: {
  "num_xml_files": 50000,
  "num_entries": 50000,
  "num_with_existing_images_in_val_classed": 50000,
  "num_missing_images_in_val_classed": 0,
  "num_entries_with_zero_valid_boxes": 0,
  "bbox_count_per_image": {
    "min": 1,
    "max": 20,
    "mean": 1.60954,
    "median": 1.0
  },
  "bbox_area": {
    "min": 12,
    "max": 17362107,
    "mean": 71384.9956260795,
    "median": 39000.0
  }
}
